# w07_action_playbook.ipynb

## Action Playbook — User Intent Lane

This notebook turns my validated model output into a content action playbook.

## 1. Ranked Actions + Reason Codes

Based on my model's findings (Precision@50: 0.760), I recommend the following actions:

| Rank | Action | Reason Code | Signal |
|------|--------|-------------|--------|
| 1 | **Refresh** | `LOW_ENGAGEMENT` | High CTR + low engagement_rate — users click but don't stay |
| 2 | **Rewrite** | `LOW_CTR_HIGH_POSITION` | Low CTR + high avg_position — good ranking but poor click-through |
| 3 | **Monitor** | `IMPROVING_ENGAGEMENT` | Engagement_rate is increasing — potential to grow |
| 4 | **Protect** | `HIGH_ENGAGEMENT` | High CTR + high engagement_rate — already performing well |
| 5 | **Review** | `LOW_IMPRESSIONS` | Low impressions — not enough data to act reliably |

### Action Definitions

**Refresh:** Update the content (new information, better formatting, fresh examples).

**Rewrite:** Re-write the content from scratch with a new angle.

**Monitor:** Check back in 30 days to see if the trend continues.

**Protect:** Keep as-is, optimize other elements (metadata, internal links).

**Review:** Collect more data before making a decision.

## 2. Intended Use and Limits

### Intended Use
This playbook is designed to help content teams prioritize which pages need attention. It should be used as a decision-support tool, not a replacement for human judgment.

### When to Use
- When you have a large content inventory and need to prioritize
- When you want to identify underperforming pages
- When you want to protect pages that are already working

### When NOT to Use
- Do NOT use for brand-new content (less than 30 days old)
- Do NOT use for pages with less than 10 impressions (not enough data)
- Do NOT use as the sole decision-maker — human review is required
- Do NOT use for automated content changes

### Key Limitations
1. **Proxy target:** Engagement is a proxy for intent, not a direct measure
2. **Sample bias:** June 2026 only — may not generalize
3. **Client-holdout:** May reduce sample size and statistical power
4. **No causal claims:** Results are directional, not causal

## 3. Human Review + The No-Go List

### What a Human MUST Check Before Acting

| Item | Why |
|------|-----|
| **Does the recommendation make sense?** | The model can be wrong — a human should verify |
| **Is the page seasonal?** | Some content naturally performs poorly at certain times |
| **Is the page new?** | New content may not have enough data yet |
| **Are there external factors?** | Algorithm updates, market changes, etc. |

### The No-Go List (What Should NEVER Be Automated)

| Action | Why Not |
|--------|---------|
| **Publishing** | Human judgment required for quality |
| **Deleting content** | The model may misclassify — deletion is irreversible |
| **Making changes to high-value pages** | Risk of harming revenue or traffic |
| **Changes to brand/voice-critical pages** | The model doesn't understand brand voice |

### What the Model CAN Help With

| Action | Why |
|--------|-----|
| **Prioritization** | Which pages to review first |
| **Suggestion** | What type of action might be appropriate |
| **Monitoring** | Flagging pages that need human attention |

## 4. Monitoring / Retrain Triggers

### When to Retrain

| Trigger | Frequency |
|---------|-----------|
| **New data available** | Monthly |
| **Performance drops** | When Precision@50 falls below 0.700 |
| **Significant data drift** | When distribution of key features changes |
| **New features available** | When new signals become available |

### What to Monitor

| Metric | Target | Alert |
|--------|--------|-------|
| Precision@50 | > 0.700 | Drop below 0.650 |
| Recall@50 | > 0.500 | Drop below 0.400 |
| Feature importance | Stable | Major shifts in top features |
| Data volume | > 10,000 pages | Significant drop in data availability |

### Monitoring Process
1. Run model weekly on latest data
2. Log performance metrics
3. If metrics drop below targets, retrain with new data
4. Review feature importance quarterly
5. If features drift significantly, re-evaluate features

## 5. Exports for the Paper

This section exports the ranked queue and metrics for your research paper.

In [ ]:
# Export ranked queue
import os
import pandas as pd
import json

# Create output directories
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# Sample queue for export
queue_data = con.sql(f"""
    SELECT 
        content_hash_id,
        AVG(gsc_avg_position) AS avg_position,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr,
        AVG(ga4_engaged_sessions) AS engagement_rate,
        SUM(gsc_impressions) AS impressions_90d
    FROM {SAMPLE}
    WHERE report_date = '2026-06-01'
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) >= 10
""").df()

# Add actions based on signals
def assign_action(row):
    if row['ctr'] > 0.05 and row['engagement_rate'] < 0.3:
        return 'REFRESH', 'LOW_ENGAGEMENT'
    elif row['ctr'] < 0.02 and row['avg_position'] < 10:
        return 'REWRITE', 'LOW_CTR_HIGH_POSITION'
    elif row['engagement_rate'] > 0.5:
        return 'PROTECT', 'HIGH_ENGAGEMENT'
    elif row['impressions_90d'] < 50:
        return 'REVIEW', 'LOW_IMPRESSIONS'
    else:
        return 'MONITOR', 'IMPROVING_ENGAGEMENT'

queue_data['action'], queue_data['reason_code'] = zip(*queue_data.apply(assign_action, axis=1))

# Save queue
queue_data.to_csv('work/outputs/action_queue.csv', index=False)
print("✅ Queue exported to work/outputs/action_queue.csv")

# Save metrics
metrics = {
    'model': 'Random Forest',
    'precision_at_50': 0.760,
    'baseline_precision': 0.680,
    'improvement': 0.080,
    'improvement_percent': 11.8,
    'validation_split': 'client-holdout'
}

with open('work/outputs/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("✅ Metrics exported to work/outputs/metrics.json")

# Export sample of queue
print("\nSample of action queue:")
display(queue_data.head(10))

## 6. Self-Check

✅ I've defined ranked actions with reason codes

✅ I've explained intended use and limits

✅ I've specified what must be human-reviewed

✅ I've named the no-go list (what should NOT be automated)

✅ I've defined monitoring and retrain triggers

✅ I've exported the queue and metrics to work/outputs/

✅ I've kept the plan practical and non-production